In [1]:
from pprint import pprint
import torch
from torch.nn import Module
from torch.export import export
from tvm.relax.frontend.torch.fx_translator import TorchFXImporter, from_fx

In [67]:
import torch
import torch.fx
from torch.export import export
from functorch.experimental.control_flow import cond

# Simple module for demonstration
class M(torch.nn.Module):
    def __init__(self) -> None:
        super().__init__()
        self.conv = torch.nn.Conv2d(
            in_channels=3, out_channels=16, kernel_size=3, padding=1
        )
        self.relu = torch.nn.ReLU()
        self.maxpool = torch.nn.MaxPool2d(kernel_size=3)

    def forward(self, x: torch.Tensor, *, constant=None) -> torch.Tensor:
        a = self.conv(x)
        a.add_(constant)
        return self.maxpool(self.relu(a)), a

example_args = (torch.randn(1, 3, 256, 256),)
example_kwargs = {"constant": torch.ones(1, 16, 256, 256)}

exported_program: torch.export.ExportedProgram = export(
    M(), args=example_args, kwargs=example_kwargs
)
#print(exported_program)
#print(exported_program.graph)
#print(exported_program.graph_signature)

for spec in exported_program.graph_signature.input_specs:
    print(spec)

InputSpec(kind=<InputKind.PARAMETER: 2>, arg=TensorArgument(name='p_conv_weight'), target='conv.weight', persistent=None)
InputSpec(kind=<InputKind.PARAMETER: 2>, arg=TensorArgument(name='p_conv_bias'), target='conv.bias', persistent=None)
InputSpec(kind=<InputKind.USER_INPUT: 1>, arg=TensorArgument(name='x'), target=None, persistent=None)
InputSpec(kind=<InputKind.USER_INPUT: 1>, arg=TensorArgument(name='constant'), target=None, persistent=None)


In [13]:
import torch
import torch.fx
class BatchNorm2d(torch.nn.Module):
    def __init__(self):
        super().__init__()
        self.bn = torch.nn.BatchNorm2d(3)

    def forward(self, input):
        return self.bn(input)

example_args = (torch.randn(1, 3, 10, 10, dtype=torch.float32),)
exported_program: torch.export.ExportedProgram = torch.export.export(
    BatchNorm2d(), args=example_args
)

graph_module = exported_program.graph_module
graph: torch.fx.graph.Graph = exported_program.graph
ops_in_model = set()
for node in graph.nodes:
    print(f"op_name: {node.op}, target: {node.target}, {type(node.target)}")
    if node.op == "output":
        print(node.args)
    if node.op == "call_function":
        ops_in_model.add(node.target.__name__)
print(f"ops in model: {ops_in_model}")

op_name: placeholder, target: p_bn_weight, <class 'str'>
op_name: placeholder, target: p_bn_bias, <class 'str'>
op_name: placeholder, target: b_bn_running_mean, <class 'str'>
op_name: placeholder, target: b_bn_running_var, <class 'str'>
op_name: placeholder, target: b_bn_num_batches_tracked, <class 'str'>
op_name: placeholder, target: input, <class 'str'>
op_name: call_function, target: aten.add.Tensor, <class 'torch._ops.OpOverload'>
op_name: call_function, target: aten._native_batch_norm_legit_functional.default, <class 'torch._ops.OpOverload'>
op_name: call_function, target: <built-in function getitem>, <class 'builtin_function_or_method'>
op_name: call_function, target: <built-in function getitem>, <class 'builtin_function_or_method'>
op_name: call_function, target: <built-in function getitem>, <class 'builtin_function_or_method'>
op_name: output, target: output, <class 'str'>
((getitem_3, getitem_4, add, getitem),)
ops in model: {'getitem', '_native_batch_norm_legit_functional.def

In [17]:
import torch
import torch.fx
import torchvision
from torchvision.models import get_model, get_model_weights, list_models
from torchvision.io import read_image
from typing import Set

def list_ops_in_graph(graph: torch.fx.Graph) -> Set:
  ops_in_model = set()
  for node in graph.nodes:
      if node.op == "call_function":
          ops_in_model.add(node.target.__name__)
  return ops_in_model

classification_models = list_models(torchvision.models)
ops_in_classification_models = set()
for model_name in classification_models:
  print(f"Model: {model_name}")
  image_tensor = read_image("dog1.jpg")
  model = get_model(model_name, weights="DEFAULT").eval()
  weights = get_model_weights(model_name).DEFAULT
  transforms = weights.transforms()

  batch = transforms(image_tensor).unsqueeze(0)
  example_args = (batch,)

  exported_program: torch.export.ExportedProgram = torch.export.export(
      model, args=example_args
  )
  del model
  del weights

  exported_program.run_decompositions()
  ops = list_ops_in_graph(exported_program.graph)
  ops_in_classification_models |= ops
  print(f"  ops: {ops}")

print(f"Ops in classification models: {ops_in_classification_models}")

# print(exported_program)

Model: alexnet
  ops: {'linear.default', 'max_pool2d.default', 'conv2d.default', 'adaptive_avg_pool2d.default', 'view.default', 'relu.default', 'dropout.default'}
Model: convnext_base
  ops: {'linear.default', 'conv2d.default', 'adaptive_avg_pool2d.default', 'add.Tensor', 'view.default', 'gelu.default', 'layer_norm.default', 'permute.default', 'mul.Tensor'}
Model: convnext_large
  ops: {'linear.default', 'conv2d.default', 'adaptive_avg_pool2d.default', 'add.Tensor', 'view.default', 'gelu.default', 'layer_norm.default', 'permute.default', 'mul.Tensor'}
Model: convnext_small
  ops: {'linear.default', 'conv2d.default', 'adaptive_avg_pool2d.default', 'add.Tensor', 'view.default', 'gelu.default', 'layer_norm.default', 'permute.default', 'mul.Tensor'}
Model: convnext_tiny
  ops: {'linear.default', 'conv2d.default', 'adaptive_avg_pool2d.default', 'add.Tensor', 'view.default', 'gelu.default', 'layer_norm.default', 'permute.default', 'mul.Tensor'}
Model: densenet121
  ops: {'linear.default', 'c

: 